# HW1：从零开始构建三层神经网络分类器

GitHub Repository: https://github.com/adehuang41/cv-hw1.git

Trained Model Weights: https://drive.google.com/file/d/1WgLlZ9yD3csx1ztqtvwvrn2S1ItNH2iw/view?usp=sharing

## 任务目标

本实验使用 `NumPy` 手工实现多层感知机（MLP），在 Fashion-MNIST 数据集上完成 10 类服饰图像分类。

## 本报告包含的内容

- 数据加载与预处理
- 模型结构与核心实现模块
- 超参数搜索说明与已有搜索结果
- 最终模型训练曲线
- 测试集准确率与混淆矩阵
- 第一层权重可视化
- 错例分析与原因讨论
- 总结与后续改进方向

为保证导出 HTML/PDF 时格式稳定，本报告不保留长代码输出，而是将关键配置、结果和图像整理为报告格式。完整可运行代码见 `final version.ipynb` 和 `src/`。

## 1. 数据加载与预处理

训练集与验证集按照 `8:2` 划分。输入图像被展平成 `784` 维向量并归一化到 `[0, 1]`，标签使用 one-hot 编码，以便后续交叉熵损失函数计算。

| 数据划分 | X 维度 | Y 维度 | 用途 |
|---|---|---|---|
| 训练集 | `(48000, 784)` | `(48000, 10)` | 模型参数学习 |
| 验证集 | `(12000, 784)` | `(12000, 10)` | 超参数选择、早停和模型选择 |
| 测试集 | `(10000, 784)` | `(10000, 10)` | 最终泛化性能评估 |

Fashion-MNIST 共包含 10 类服饰图像：`T-shirt/top`、`Trouser`、`Pullover`、`Dress`、`Coat`、`Sandal`、`Shirt`、`Sneaker`、`Bag`、`Ankle boot`。

## 2. 模型结构与实现

最终模型使用三层结构（输入层 + 两层隐藏层 + 输出层），激活函数为 ReLU，不使用 Dropout。输出层使用 softmax，损失函数为交叉熵。优化器使用 SGD + Momentum，并配合指数学习率衰减。

| 模块 | 配置 |
|---|---|
| 输入层 | 784 |
| 隐藏层 1 | 256, ReLU |
| 隐藏层 2 | 128, ReLU |
| 输出层 | 10, Softmax |
| 损失函数 | Cross-Entropy Loss |
| 优化器 | SGD, momentum=0.9 |
| 学习率 | 0.01 |
| 学习率衰减 | exponential decay, decay_rate=0.9, decay_steps=5 |
| L2 正则化 | weight_decay=1e-4 |
| Batch size | 64 |
| Dropout | 0.0 |
| Early stopping | patience=5, min_delta=1e-4 |

核心代码按功能拆分在 `src/` 中：`data_utils.py` 负责数据读取、归一化和 batch 迭代；`layers.py`、`activations.py`、`model.py` 负责网络层、激活函数和 MLP 前向/反向传播；`loss.py` 实现交叉熵；`optimizers.py` 实现 SGD 与学习率调度；`train.py` 实现训练、验证、保存最佳权重和早停；`search.py` 实现超参数搜索；`test.py` 实现测试集评估、混淆矩阵、权重可视化和错例分析。

## 3. 超参数搜索

搜索空间覆盖隐藏层维度、学习率、L2 正则强度、batch size、激活函数和 dropout。为了避免每次打开 notebook 都重新跑完整网格搜索，`final version.ipynb` 默认复用已有搜索报告；如果需要完整复现，可将 `RUN_GRID_SEARCH` 改为 `True`。

```python
param_grid = {
    'hidden_dims': [[128, 64], [256, 128]],
    'learning_rate': [0.1, 0.05, 0.01],
    'weight_decay': [1e-4, 0.0],
    'batch_size': [64, 128],
    'activation': ['relu', 'sigmoid', 'tanh'],
    'dropout_rate': [0.0, 0.2]
}
```

网格搜索共覆盖 144 组配置。Top-5 结果如下：

| Rank | Val Acc | hidden_dims | learning_rate | weight_decay | batch_size | activation | dropout_rate |
|---|---:|---|---:|---:|---:|---|---:|
| Top 1 | 0.8862 | [256, 128] | 0.01 | 0.0001 | 64 | relu | 0.0 |
| Top 2 | 0.8862 | [256, 128] | 0.01 | 0.0 | 64 | relu | 0.0 |
| Top 3 | 0.8850 | [128, 64] | 0.05 | 0.0 | 128 | tanh | 0.0 |
| Top 4 | 0.8829 | [256, 128] | 0.01 | 0.0 | 64 | tanh | 0.0 |
| Top 5 | 0.8824 | [128, 64] | 0.01 | 0.0001 | 64 | relu | 0.0 |

从已有搜索结果可以看到，`[256, 128] + relu + lr=0.01 + weight_decay=0.0001 + dropout=0.0 + batch_size=64` 的组合取得了最高验证集准确率（0.8862），因此使用该组合训练最终模型。

## 4. 最终模型训练

最终模型训练最多进行 30 轮，早停机制在验证集准确率连续 5 轮无显著提升时自动终止，并保存验证集表现最佳的模型权重。实际训练在第 25 轮早停，最佳验证集准确率为 **0.8963**，出现在第 **20** 轮。最终一轮训练集准确率为 **0.9405**，验证集准确率为 **0.8913**。

训练过程中记录训练集和验证集的 Loss 曲线，以及验证集 Accuracy 曲线。下图同时展示训练集 Accuracy，便于观察训练/验证差距。

![训练集与验证集 Loss / Accuracy 曲线](logs/report_training_curves.png)

曲线显示，训练 Loss 持续下降，说明模型能够有效拟合训练数据；验证 Loss 在第 16 至 20 轮附近达到较低水平，之后开始波动并略有上升，说明继续训练会带来一定过拟合风险。验证集 Accuracy 在第 20 轮达到最高值，因此最终保存该轮对应的模型权重。

### 4.1 逐轮训练记录

| Epoch | Train Loss | Val Loss | Train Acc | Val Acc |
|---:|---:|---:|---:|---:|
| 1 | 0.5734 | 0.4411 | 0.7989 | 0.8411 |
| 2 | 0.4042 | 0.3934 | 0.8567 | 0.8591 |
| 3 | 0.3620 | 0.3865 | 0.8688 | 0.8640 |
| 4 | 0.3360 | 0.4212 | 0.8779 | 0.8479 |
| 5 | 0.3184 | 0.3349 | 0.8830 | 0.8761 |
| 6 | 0.2973 | 0.3289 | 0.8904 | 0.8824 |
| 7 | 0.2860 | 0.3166 | 0.8948 | 0.8872 |
| 8 | 0.2724 | 0.3264 | 0.8998 | 0.8810 |
| 9 | 0.2648 | 0.3134 | 0.9033 | 0.8856 |
| 10 | 0.2576 | 0.3142 | 0.9041 | 0.8838 |
| 11 | 0.2445 | 0.3090 | 0.9104 | 0.8852 |
| 12 | 0.2367 | 0.3074 | 0.9132 | 0.8893 |
| 13 | 0.2307 | 0.3315 | 0.9141 | 0.8757 |
| 14 | 0.2247 | 0.3035 | 0.9172 | 0.8871 |
| 15 | 0.2204 | 0.3047 | 0.9186 | 0.8878 |
| 16 | 0.2089 | 0.2914 | 0.9232 | 0.8940 |
| 17 | 0.2014 | 0.3022 | 0.9267 | 0.8924 |
| 18 | 0.1972 | 0.3054 | 0.9273 | 0.8871 |
| 19 | 0.1949 | 0.2945 | 0.9287 | 0.8940 |
| 20 **best** | 0.1887 | 0.2899 | 0.9311 | 0.8963 |
| 21 | 0.1799 | 0.2954 | 0.9336 | 0.8950 |
| 22 | 0.1768 | 0.3074 | 0.9349 | 0.8924 |
| 23 | 0.1720 | 0.3019 | 0.9369 | 0.8906 |
| 24 | 0.1674 | 0.3088 | 0.9389 | 0.8910 |
| 25 | 0.1634 | 0.3146 | 0.9405 | 0.8913 |

## 5. 测试集评估与可视化

下面加载保存的最佳权重，在独立测试集上计算准确率，并展示混淆矩阵和第一层权重可视化结果。

| 指标 | 数值 |
|---|---:|
| Test Loss | 0.3205 |
| Test Accuracy | 0.8874 |

测试集准确率为 **88.74%**，说明从零实现的三层 MLP 已经能够在 Fashion-MNIST 上达到较稳定的分类效果。

![测试集混淆矩阵](logs/confusion_matrix.png)

混淆矩阵显示，模型在 `Trouser`、`Sandal`、`Sneaker`、`Bag`、`Ankle boot` 等轮廓差异较明显的类别上表现较稳定；主要错误集中在视觉形态相似的上衣类之间，例如 `Pullover`、`Coat`、`Shirt` 和 `T-shirt/top`。这与 Fashion-MNIST 低分辨率灰度图像的特点一致：领口、开襟线、纽扣等细节在 28×28 图像中很容易丢失。

## 6. 权重可视化与空间模式分析

第一隐藏层权重矩阵形状为 `(784, 256)`，将每列还原为 28×28 图像可以观察每个神经元的空间偏好。以下以前 16 个神经元为例进行讨论。

![第一层权重可视化](logs/weight_visualization.png)

**整体特征**

所有神经元的权重图均呈现出分散的高低权重像素交错分布，整体噪声感较强，缺少清晰的块状边缘结构。这与本次训练正则化强度极低（weight_decay=1e-4, dropout=0.0）有关：1e-4 的权重衰减对权重幅度的约束十分有限，MLP 仍然倾向于让权重分散到大量像素上，而非集中到少数有区分力的感受野，导致可视化结果较难解读出明确的语义。

**具体模式**

- **孤立高激活锚点型**：Neuron #2（上方）、#3（左上方）、#5（左侧中部）、#16（左侧中部）在整体蓝绿色背景中均出现清晰孤立的高亮黄色像素点，可能对应训练集中特定类别在固定坐标频繁出现的纹理锚点，但因孤立性强、不成连续结构而泛化能力有限；其中 #5 同时伴有多处深紫色低权重像素，高低权重并存，结构最为复杂。

- **均匀分散型**：Neuron #8、#10 以均匀蓝绿色为主，无显著高亮或暗区聚集，权重散布全图，是最难解读语义的一类。Neuron #4、#6、#11 的背景色调偏向鲜亮的黄绿色，整体色彩明亮，与其他神经元的蓝绿背景不同。

- **整体偏暗型**：Neuron #1、#7、#13 的背景整体呈深蓝/蓝紫色，全局权重幅度系统性偏低，是 16 个神经元中整体色调最暗的一组，暗示对较大范围像素区域的广泛抑制响应；Neuron #3、#12 也散布着较多深紫色低权重像素，但暗点相对分散，未形成连片聚集。

**结论**

在极弱正则化约束下，第一层权重未能收敛到具有清晰空间语义的检测器，整体停留在分散的统计相关模式。这从侧面说明：在全连接 MLP 中，适当加大权重衰减或引入 Dropout 有助于迫使每个神经元聚焦于更少、更有意义的输入维度，从而产生更可解释的权重可视化结果。

## 7. 错例分析

![错例分析](logs/error_analysis.png)

模型在测试集上共误分类 **1126 / 10000（11.3%）** 张图像。下表为混淆矩阵中 Top-5 混淆对：

| 真实类别 | 预测类别 | 错误数 |
|---|---|---:|
| Pullover | Coat | 127 |
| T-shirt/top | Shirt | 125 |
| Shirt | T-shirt/top | 105 |
| Pullover | Shirt | 71 |
| Shirt | Coat | 67 |

Top-5 全部集中于上装四类（T-shirt/top、Shirt、Pullover、Coat），合计 495 例，占总误分类数的 **44%**。以下结合随机抽取的 9 个错例，对主要混淆模式进行分析。

**上装类别互混（占误分类主体）**

- **Pullover → Coat**（最多，127 例）：套头衫与外套的核心区别在于领口结构（圆领 vs 翻领/开襟）和袖管宽度比例。在 28×28 灰度图中，这些细节往往压缩至 1–2 个像素宽的灰度过渡，几乎无法被全连接层可靠感知。版型宽大或颜色偏深的套头衫尤其容易与轮廓相近的外套混淆。

- **T-shirt/top ↔ Shirt 双向混淆**（125 + 105 = 230 例）：T 恤与衬衫的整体廓形几乎完全一致，均为上宽下窄的矩形主体区域。衬衫的关键辨别特征（门襟线、纽扣排列、领口细节）在低分辨率灰度图中极难察觉，导致两类之间发生大量对称性互混。领口偏平的 T 恤容易被判为衬衫，而轮廓宽松的衬衫则容易被判为 T 恤。

- **Pullover → Shirt**（71 例）：低领或 V 领套头衫的领口较平整，整体轮廓与平口衬衫接近，模型在无法感知材质和领口细节的情况下产生混淆。

- **Shirt → Coat**（67 例）：版型宽大的夹克式衬衫肩线外扩、下摆较宽，在像素分布上与外套的廓形相近，被误判为外套。

**随机错例中的采样分布与共性规律**

本次随机抽取的 9 张错例中，**5 张为鞋类混淆**（Sandal→Sneaker、Sandal→Ankle boot、Sneaker→Ankle boot），**4 张为上装混淆**（Shirt→Coat、Pullover→Coat、Shirt→Dress）。

两类错误的成因不同：

- **鞋类混淆**：Sandal 的镂空结构和细带特征在 28×28 灰度图中被压缩至几个像素，侧视剪影与同高度的运动鞋或短靴几乎无法区分；高帮运动鞋覆盖脚踝，侧视轮廓高度与 Ankle boot 相近，极易混淆。
- **上装混淆**：领口细节、开襟线、袖型等关键辨别特征在低分辨率下被压缩，整体廓形相近的类别之间发生混淆，与 Top-5 分析一致。

**总结**

本次运行的主要系统性误差集中于上装类别内部（Top-5 全为上装混淆），根本原因在于 MLP 缺乏局部特征提取机制，无法捕捉领口形状、开襟方式等细节。引入卷积结构（CNN）或注意力机制将是改善此类混淆的最直接方向。

## 8. 结论

综合训练曲线、测试集结果、混淆矩阵和错例分析，可以得到以下结论：

1. 从零实现的 MLP 能够在 Fashion-MNIST 上稳定达到较高准确率（测试集 88.74%，最优验证集 89.63%），证明前向传播、反向传播、交叉熵、SGD+Momentum 和学习率衰减模块工作正常。
2. 轮廓差异明显的类别（如 `Trouser`、`Bag`、`Ankle boot`）更容易区分；上装类别（T-shirt/top、Shirt、Pullover、Coat）之间由于在 28×28 灰度图中廓形高度相似，是主要错误来源，占总误分类数的 44%。
3. 第一层权重在极弱正则化（weight_decay=1e-4）下呈现出分散的统计相关模式，尚未形成具有清晰空间语义的局部检测器。
4. 如果继续优化，可以尝试引入卷积结构（CNN）以提取局部特征、加大正则化强度、或通过更精细的超参数搜索进一步降低上装类别之间的混淆。